# Deep ML Starter Notebook
This notebook verifies the installations of `PyTorch`, `TensorFlow`, `Keras`, `Lightning`, `Transformers` and tests GPU detection and MLflow connectivity.

In [ ]:
import torch
import tensorflow as tf
import keras
import lightning as L
import transformers
import mlflow

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Transformers version: {transformers.__version__}")

In [ ]:
# Verify GPU accessibility
cuda_available = torch.cuda.is_available()
print(f"[PyTorch] CUDA available: {cuda_available}")
if cuda_available:
    print(f"[PyTorch] GPU Name: {torch.cuda.get_device_name(0)}")

gpus = tf.config.list_physical_devices('GPU')
print(f"[TensorFlow] GPUs detected: {len(gpus)}")
for gpu in gpus:
    print(f"  - {gpu}")

In [ ]:
# Quick PyTorch training check and MLflow log
import torch.nn as nn
import torch.optim as optim

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("deep-ml-starter")

# Simple Model
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)
    def forward(self, x):
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinearModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Dummy data
x_dummy = torch.randn(100, 10).to(device)
y_dummy = torch.randn(100, 1).to(device)

with mlflow.start_run():
    # Run a quick training loop
    for epoch in range(5):
        optimizer.zero_grad()
        outputs = model(x_dummy)
        loss = criterion(outputs, y_dummy)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/5 - Loss: {loss.item():.4f}")
        mlflow.log_metric("loss", loss.item(), step=epoch)
        
    # Log dummy hyperparameter
    mlflow.log_param("epochs", 5)
    mlflow.log_param("device", str(device))
    
    # Log the PyTorch model
    mlflow.pytorch.log_model(model, "model")
    print("Logged run and PyTorch model to MLflow successfully!")